In [29]:
library(auk)
library(tidyverse)

library(magrittr)
library(httr)
library(data.table)
library(readr)

library(lubridate)
library(hms)

readRenviron("~/.Renviron")
R.home()

[1] "/usr/lib/R"

In [30]:
PROJECT_DIR = getwd()

ebd_dir = file.path(PROJECT_DIR, "source_data", "ebd_relMar-2026")
ebd_filename = "ebd_relMar-2026.txt"

sed_dir = file.path(PROJECT_DIR, "source_data", "ebd_sampling_relMar-2026")
sed_filename = "ebd_sampling_relMar-2026.txt"

In [31]:
auk_version()
auk_ebd_version(file.path(ebd_dir, ebd_filename))

$auk_version
[1] "auk 0.9.0"

$ebd_version
[1] "2025-10-28"

$taxonomy_version
[1] 2025

$ebd_version
[1] "2026-03-01"

$taxonomy_version
[1] 2025

In [ ]:
# source("ebd_functions.R")
# source("land_cover_functions.R")
# source("climate_functions.R")
# dist_to_coast_df = read_in_dist_to_coast_data(file.path(PROJECT_DIR, "source_data", "NASA_dist2coast.txt"))

<span style="color:deepskyblue">Citation:  
*eBird Basic Dataset. Version: EBD_relSep-2025. Cornell Lab of Ornithology, Ithaca, New York. Sep 2025.*</span>

### Data extraction

#### Pileated Woodpecker, 2024, SC

In [ ]:
file.desc <- "pileated_wp_20251130"
f_out_ebd_only <- file.path(PROJECT_DIR, "output", "auk", paste0("ebd_", file.desc, ".txt"))
f_out_sed_only <- file.path(PROJECT_DIR, "output", "auk", paste0("sed_", file.desc, ".txt"))

# Full dataset filtering
f_in_ebd <- file.path(ebd_full_dir, ebd_full_filename)
f_in_sed <- file.path(sed_full_dir, sed_full_filename)
ebd_filters <- auk_ebd(f_in_ebd, file_sampling = f_in_sed) %>%
    auk_species("Pileated Woodpecker") %>%
    auk_country("United States") %>%
    auk_state("US-SC") %>% 
    auk_complete() %>%
    auk_date(c("2024-01-01", "2024-12-31"))
# filters: https://cornelllabofornithology.github.io/auk/reference/index.html#section-filter

auk_filter(ebd_filters, file = f_out_ebd_only, file_sampling = f_out_sed_only, overwrite = TRUE)

#### Read data back in

In [ ]:
ebd_only_df = read_ebd(f_out_ebd_only, unique=TRUE, rollup=TRUE) # do not need to use auk_unique() when unique=TRUE passed here. Same for auk_rollup()
sed_only_df = read_sampling(f_out_sed_only, unique=TRUE)

### Zero-fill and clean

In [ ]:
ebd_zf_df <- auk_zerofill(ebd_only_df, sampling_events = sed_only_df) %>%
    collapse_zerofill()

In [ ]:
# Convert Xs to N/A. Set distance to 0 for stationary checklists 
ebd_zf_df <- mutate(ebd_zf_df,
                    observation_count = if_else(observation_count == "X", 
                                                NA_character_, observation_count),
                    observation_count = as.integer(observation_count),
                    effort_distance_km = if_else(protocol_name == "Stationary", 
                                                 0, effort_distance_km))
table(ebd_zf_df$species_observed)
# Reduce effort variability
ebd_zf_df <- ebd_zf_df %>% 
  filter(duration_minutes <= 60 * 5,
         effort_distance_km <= 5,
         number_observers <= 5)
        #  number_observers <= 10)
# https://strimas.com/ebp-workshop/presabs.html
table(ebd_zf_df$species_observed)

In [ ]:
str(ebd_zf_df)

In [ ]:
ebd_zf_df_file = file.path(PROJECT_DIR, "output", paste0(file.desc, "_zf.csv"))
write_csv(ebd_zf_df, ebd_zf_df_file)

### Filter

In [ ]:
# function to convert time observation to hours since midnight
time_to_decimal <- function(x) {
  x <- as_hms(x)
  hour(x) + minute(x) / 60 + second(x) / 3600
}

ebd_zf_df <- ebd_zf_df |> 
  mutate(
    # convert count to integer and X to NA
    # ignore the warning "NAs introduced by coercion"
    observation_count = as.integer(observation_count),
    # effort_distance_km to 0 for stationary counts
    effort_distance_km = if_else(protocol_name == "Stationary", 
                                 0, effort_distance_km),
    # convert duration to hours
    effort_hours = duration_minutes / 60,
    # speed km/h
    effort_speed_kmph = effort_distance_km / effort_hours,
    # convert time to decimal hours since midnight
    hours_of_day = time_to_decimal(time_observations_started),
    # split date into year and day of year
    year = year(observation_date),
    day_of_year = yday(observation_date)
  )

ebd_zf_df_filtered <- ebd_zf_df |> 
  filter(protocol_name %in% c("Stationary", "Traveling"),
         effort_hours <= 6,
         effort_hours >= 15/60, # added minimum duration
         effort_distance_km <= 10,
         effort_speed_kmph <= 100,
         number_observers <= 10) # (already filtered to <=5 above)
# https://ebird.github.io/ebird-best-practices/ebird.html#sec-ebird-effort